# llm-inference-bench: Kaggle T4 runner

Thin wrapper for running the sweep on Kaggle's free T4 (30 hr/week, 16 GB).

**Before running**: in the notebook sidebar, enable GPU (T4 x2 or T4) and Internet access.

What this notebook does:
1. Clones the repo
2. Installs the GPU dependency stack
3. Runs the sweep with `--runner vllm --hardware T4`
4. Aggregates results and zips them for download

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO_URL = 'https://github.com/<your-user>/llm-inference-bench.git'
os.makedirs('/kaggle/working', exist_ok=True)
%cd /kaggle/working
if not os.path.isdir('llm-inference-bench'):
    !git clone {REPO_URL}
%cd llm-inference-bench

In [ ]:
!pip install -q -e .
!pip install -q -r requirements-gpu.txt

In [ ]:
# Sanity: smoke run with MockRunner — should finish in seconds.
!python -m llm_bench run --config configs/smoke.yaml --runner mock --output results/runs --hardware T4

In [ ]:
# The real sweep. Expect long-running cells; use --skip-failed to keep going past OOMs.
!python -m llm_bench sweep --config configs/sweep.yaml --runner vllm --output results/runs --hardware T4

In [ ]:
!python -m llm_bench analyze --input results/runs --output results/summary
!ls -la results/summary

In [ ]:
# Zip everything for download via the Kaggle output panel.
import shutil
shutil.make_archive('/kaggle/working/bench-results', 'zip', 'results')